In [34]:
import copy
import tiktoken

#from settings import Settings
#from models import Question
from src.services.OpenAIClient import (
    OpenAIClient,
    Gpt4o,
    Gpt4oMini,
    #Gpto1Mini,
    #Gpto1Preview,
    Gpto1,
    Gpto3,
    Gpt5,
)

from src.services.accuracy import calculate_accuracy
from src.services.read_excel import read_excel
##from src.services.recorder import ApiHistoryRecorder
from src.strategies.solve_questions_by_openai import (
    BasicSolveQuestionPrompt,
    solve_questions_by_openai_independently,
    solve_type_d_questions_by_openai_dependently,
    OptimizedSolveQuestionPrompt1,
    NormalSolveQuestionPrompt,
    OptimizedSolveQuestionPrompt2_for_essential_a_b,
    OptimizedSolveQuestionPrompt2_for_c_d
)
from src.strategies.translate_to_English_by_openai import (
    BasicTranslateToEnglishPrompt,
    OptimizedTranslateToEnglishPrompt1,
    NormalTranslateToEnglishPrompt,
    OptimizedTranslateToEnglishPrompt,
)

In [35]:
record_excel_path = "../record/record.xlsx"
#file_path = "../problems/76th/essential"
file_path = "../problems/76th/academic_d"

problem_file_path = f'{file_path}/questions.xlsx'
answer_file_path = f'{file_path}/answers.xlsx'

### =====変更=====
model = Gpt5

is_solving_prompt_normal = True

EXECUTE_NUM = 1

is_translated_to_English = False

solve_question_prompt = NormalSolveQuestionPrompt if is_solving_prompt_normal else OptimizedSolveQuestionPrompt2_for_c_d # OptimizedSolveQuestionPrompt1
translate_to_english_prompt = OptimizedTranslateToEnglishPrompt

is_image_contained = True
### ============

WAIT_SEC = 15
solve_num = -1
batch_size = 10


is_dry_run = False

is_d = file_path[-1] == 'd'


assert is_image_contained == (file_path[-1] in {'c', 'd'})
assert EXECUTE_NUM <= 3



In [36]:
#api_history_recorder = ApiHistoryRecorder(excel_path=record_excel_path)
openai_client = OpenAIClient(
    model=model,
    #api_history_recorder=api_history_recorder,
)

In [37]:
questions = read_excel(file_path=file_path)

for i in range(4):
    print(f'======question:{i+1}======')
    print(questions[i].type_d_common_sentence)
    print(questions[i].question_sentence)
    print(questions[i].answer_options)
    print(questions[i].correct_answer)
    print(questions[i].image_path)


======question:1======
犬、ペキニーズ、雄、8歳齢。左右対称性の貧毛がみられた。〔図1−A〕は貧毛部の皮膚、〔図1−B〕は同部位の病理組織像（HE染色）である。
適切な組織所見はどれか。
１．痂皮の形成 ２．表皮の錯角化 ３．膿疱の形成 ４．真皮の石灰沈着 ５．立毛筋の肥大
{<AnswerEnum.CHOICE_4: 4>}
../problems/76th/academic_d/images/image1.PDF
======question:2======
犬、ペキニーズ、雄、8歳齢。左右対称性の貧毛がみられた。〔図1−A〕は貧毛部の皮膚、〔図1−B〕は同部位の病理組織像（HE染色）である。
本疾患の発生に最も関連するのはどれか。
１．グルココルチコイド
２．甲状腺ホルモン
３．インスリン
４．アルドステロン
５．カテコールアミン
{<AnswerEnum.CHOICE_1: 1>}
../problems/76th/academic_d/images/image1.PDF
======question:3======
犬、ヨークシャー・テリア、雄、5か月齢。同居犬と遊んでいるときにキャンと鳴いてから右前肢を挙上しているとの主訴で来院。〔図2〕は右前肢の単純X線像（A：側方像、B：頭尾像）である。
X線所見として最も適当なのはどれか。 a 上腕骨内側上顆骨折 b 上腕骨外側上顆骨折 c 成長板骨折 d モンテジア骨折 e 尺骨近位骨折
１．a, b ２．a, e ３．b, c ４．c, d ５．d, e
{<AnswerEnum.CHOICE_3: 3>}
../problems/76th/academic_d/images/image2.PDF
======question:4======
犬、ヨークシャー・テリア、雄、5か月齢。同居犬と遊んでいるときにキャンと鳴いてから右前肢を挙上しているとの主訴で来院。〔図2〕は右前肢の単純X線像（A：側方像、B：頭尾像）である。
本症例の治療として最も適当なのはどれか。
１．エーマー吊り包帯
２．ホブル（足かせ）包帯
３．骨片除去手術
４．ラグスクリューとピンによる固定
５．骨プレートによる固定
{<AnswerEnum.CHOICE_4: 4>}
../problems/76th/academi

In [38]:
import asyncio

for num in range(EXECUTE_NUM):
    question_temp = copy.deepcopy(questions[:solve_num] if solve_num >= 0 else questions)
    if is_d:
        questions_res = await solve_type_d_questions_by_openai_dependently(
            openai_client=openai_client,
            questions=question_temp,
            is_translated_to_English=is_translated_to_English,
            excel_output_path=problem_file_path,
            solve_question_prompt=solve_question_prompt,
            translate_to_english_prompt=translate_to_english_prompt,
            is_image_contained=is_image_contained,
            batch_size=batch_size,
            is_dry_run=is_dry_run,
            does_also_write_openai_answer=True,
        )
    else:
        questions_res = await solve_questions_by_openai_independently(
            openai_client=openai_client,
            questions=question_temp,
            is_translated_to_English=is_translated_to_English,
            excel_output_path=problem_file_path,
            solve_question_prompt=solve_question_prompt,
            translate_to_english_prompt=translate_to_english_prompt,
            is_image_contained=is_image_contained,
            batch_size=batch_size,
            is_dry_run=is_dry_run,
            does_also_write_openai_answer=True,
        )
    if num < EXECUTE_NUM-1:
        await asyncio.sleep(WAIT_SEC)


[{<Roles.system: 'system'>: 'Please answer this question.'}, {<Roles.user: 'user'>: '犬、ペキニーズ、雄、8歳齢。左右対称性の貧毛がみられた。〔図1−A〕は貧毛部の皮膚、〔図1−B〕は同部位の病理組織像（HE染色）である。 適切な組織所見はどれか。 The answer options are １．痂皮の形成 ２．表皮の錯角化 ３．膿疱の形成 ４．真皮の石灰沈着 ５．立毛筋の肥大. Respond with only the number of your choice (e.g., 1, 2, 3, etc.).'}]
[{<Roles.system: 'system'>: 'Please answer this question.'}, {<Roles.user: 'user'>: '犬、ヨークシャー・テリア、雄、5か月齢。同居犬と遊んでいるときにキャンと鳴いてから右前肢を挙上しているとの主訴で来院。〔図2〕は右前肢の単純X線像（A：側方像、B：頭尾像）である。 X線所見として最も適当なのはどれか。 a 上腕骨内側上顆骨折 b 上腕骨外側上顆骨折 c 成長板骨折 d モンテジア骨折 e 尺骨近位骨折 The answer options are １．a, b ２．a, e ３．b, c ４．c, d ５．d, e. Respond with only the number of your choice (e.g., 1, 2, 3, etc.).'}]
[{<Roles.system: 'system'>: 'Please answer this question.'}, {<Roles.user: 'user'>: '犬、キャバリア・キング・チャールズ・スパニエル、避妊雌、9歳齢。頸部をしきりに掻くとの主訴で来院。〔図3〕は頸部MRI矢状断像（A：T1強調像、B：T2強調像）である。 最も疑われる疾患はどれか。 a 椎間板ヘルニア b 環椎軸椎不安定症 c 脊髄腫瘍 d 脊髄空洞症 e 後頭骨形成不全 The answer options are １．a, b ２．a, e ３．b, c ４．c, d ５．d, e. Respond with only the number of

In [39]:
Qs = questions_res if solve_num < 0 else questions_res[:solve_num]
token_sum = 0

tiktoken_model = "gpt2"

def cal_token_num(sentence):
    enc = tiktoken.get_encoding(tiktoken_model)
    tokens = enc.encode(sentence)
    return len(tokens)

for q in Qs:
    if is_translated_to_English:#not q.is_correct():
        print(q.question_sentence)
        print(q.answer_options)
        print(q.question_sentence_in_English)
        print(q.openai_answer)
        print(q.correct_answer)
        #question_token_num = cal_token_num(q.question_sentence_in_English)
        #options_token_num = cal_token_num(q.answer_options_in_English)
        #token_sum += question_token_num + options_token_num

print(token_sum)


0


In [40]:
import datetime

dt_now = datetime.datetime.now()
print(dt_now.strftime('%Y/%m/%d %H:%M'))
print(f"model:{openai_client.model.model}")
score = calculate_accuracy(questions=questions_res)
print(f"score:{score}")

2025/09/16 16:29
model:gpt-5
score:{'accuracy': 0.9166666666666666, 'correct_num': 55, 'wrong_num': 5}


In [41]:
"""
from src.services.image_encoder import pdf_encoder_in_base64
base64_image = pdf_encoder_in_base64("../problems/74th/academic_c/images/image1.PDF")

openai_client2 = OpenAIClient(model=Gpt4oMini)

answer =  await openai_client2.fetch_completion(
    system_prompt='you are vet.',
    user_prompt="""
#    これはなんの写真ですか？
""",
    base64_image=base64_image,
)

print(answer)
"""

',\n    base64_image=base64_image,\n)\n\nprint(answer)\n'

In [42]:
system_prompt = OptimizedSolveQuestionPrompt2_for_essential_a_b.system_prompt
WIDTH = 70
cnt = 0
for ch in list(system_prompt.split()):
    print(ch, end=' ')
    cnt += len(ch)
    if cnt >= WIDTH:
        cnt = 0
        print()

You are tasked with answering this veterinary question. This is an examination in Japan. 
Therefore, you must refer to the laws, guidelines, and political system in Japan. Think 
deeply and thoroughly. Choose the best possible answer from the given options. Do not 
include explanations or additional text in the output. 